# LC #242 — Valid Anagram
**Category:** HashMap / Counter | **Difficulty:** Easy | **Day 1**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>The Problem:</strong> Given two strings <code>s</code> and <code>t</code>, return <code>true</code>
if <code>t</code> is an anagram of <code>s</code>.
An anagram uses the same characters in any order.
</div>

**Examples:**
```
"anagram", "nagaram"  → true   (same letters, rearranged)
"rat",     "car"      → false  (different letters)
"listen",  "silent"   → true
```

**Core Insight:** Two strings are anagrams if and only if they have **identical character frequency maps**. Build the frequency map for both and compare — or build one and cancel-out with the other.

## Mental Models

**1. The Frequency Fingerprint**
Every string has a unique fingerprint: a map of `{char: count}`. Two strings are anagrams iff their fingerprints are identical. This is the most direct mental model — build both maps, compare.

**2. The Cancel-Out Pattern**
More elegant: increment counts for `s`, decrement for `t`. If they're anagrams, every character added by `s` is cancelled by `t`. Any non-zero count at the end means they differ. This works in a single combined map.

**3. Bounded Space Observation**
For lowercase English letters: at most 26 distinct keys in the hashmap, regardless of string length. This makes space complexity O(1) — not O(n). This is a key interview insight: "bounded alphabet = O(1) space."

In [ ]:
# Approach 1 — Sort both strings and compare
# O(n log n) time, O(n) space (sorting creates copies)

def isAnagram_sort(s: str, t: str) -> bool:
    return sorted(s) == sorted(t)

# Test
print(isAnagram_sort("anagram", "nagaram"))  # True
print(isAnagram_sort("rat", "car"))          # False

# Approach 2 — Python Counter (built-in, clean)
from collections import Counter

def isAnagram_counter(s: str, t: str) -> bool:
    return Counter(s) == Counter(t)

print(isAnagram_counter("listen", "silent"))  # True

## Trace: Cancel-Out Approach

Trace `s = "rat"`, `t = "car"`:

**Build count from `s = "rat"`:**
```
count = {'r': 1, 'a': 1, 't': 1}
```

**Cancel with `t = "car"`:**
- `c`: not in count → count['c'] = -1 → return False ✓

**Trace `s = "anagram"`, `t = "nagaram"`:**

Build from `s`: `{'a': 3, 'n': 1, 'g': 1, 'r': 1, 'm': 1}`

Cancel with `t = "nagaram"`:
- n: 1→0, a: 3→2, g: 1→0, a: 2→1, r: 1→0, a: 1→0, m: 1→0

All counts are 0 → return True ✓

In [ ]:
# Optimal — O(n) time, O(1) space (26-letter bounded hashmap)
# Manual cancel-out — most instructive for interviews.

def isAnagram(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False          # early exit — different lengths can't be anagrams

    count = {}
    for c in s:
        count[c] = count.get(c, 0) + 1   # build frequency map for s
    for c in t:
        count[c] = count.get(c, 0) - 1   # cancel out with t
        if count[c] < 0:
            return False                   # t has a char not in s (or more of it)

    return True   # all counts cancelled to 0 (or 0 remaining)

# Pythonic one-liner:
# return Counter(s) == Counter(t)

# Test
print(isAnagram("anagram", "nagaram"))  # True
print(isAnagram("rat", "car"))          # False
print(isAnagram("a", "ab"))             # False  (length check catches this instantly)

## Complexity Analysis

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Sort comparison | O(n log n) | O(n) | Creates sorted copies |
| `Counter(s) == Counter(t)` | O(n) | O(1)* | Pythonic, two passes |
| **Manual cancel-out** | **O(n)** | **O(1)*** | **Early exit on mismatch** |

*O(1) because the alphabet is bounded: at most 26 keys for lowercase English. If the alphabet were unbounded (Unicode), space would be O(k) where k = unique characters.

**Why early length check matters:** Strings of different lengths can never be anagrams. This check costs O(1) and avoids building the entire frequency map for clearly non-anagram inputs.

## Interview Q&A

**Q: What if the input contains Unicode characters?**
A: Counter and the hashmap approach both still work — keys are just Unicode code points instead of letters. Space complexity becomes O(k) where k is the number of unique characters, not O(1).

**Q: Why is `Counter` O(1) space for lowercase letters?**
A: At most 26 unique keys regardless of string length. `Counter("aaaaaaaaaaaa")` has one key: `{'a': 12}`. The map size is bounded by the alphabet, not the string length.

**Q: Your early exit checks `count[c] < 0`. Why not `!= 0`?**
A: We're cancelling as we go. A count of 0 is fine (character balanced). Negative means `t` has more of this character than `s` — that's a mismatch. Positive counts remaining at the end would also mean mismatch, but the length check prevents that case (equal lengths + no negatives = all counts are 0).

**Q: What's the difference between `Counter` and a plain `dict` for this problem?**
A: Functionally equivalent here. `Counter` handles missing keys gracefully (returns 0 for unseen keys), subtraction of two Counters drops negative/zero counts automatically. `Counter` is cleaner but knowing the manual approach shows you understand what's happening.

## The Citi Angle

**Character frequency = data fingerprinting.** The same mental model applies to data integrity checks: if you expect a set of metric names from 6,000 servers and want to verify no names changed between collection cycles, you compare frequency fingerprints.

**Practical analogy:**
```python
# Verify expected vs actual metric keys in telemetry batch
expected = Counter(["cpu_pct", "mem_pct", "disk_read", "disk_write"])
actual   = Counter([m["metric_name"] for m in batch])
if expected != actual:
    alert(f"Metric schema drift: {expected - actual} missing, {actual - expected} unexpected")
```

**Interview tie-in:** "The frequency map pattern generalizes beyond characters — at Citi I used it to fingerprint metric schemas and catch schema drift between APM tool upgrades. Same O(n) build, O(1) comparison."

## Summary

| | Value |
|--|--|
| **Pattern** | HashMap — frequency fingerprint |
| **Time** | O(n) |
| **Space** | O(1) for lowercase English (bounded alphabet) |
| **Key line** | `if count[c] < 0: return False` |
| **Say in interview** | "Frequency map. O(n) time, O(1) space because the alphabet is bounded at 26 characters." |

**The two patterns to know:**
```python
# Pattern A: Counter comparison (Pythonic)
return Counter(s) == Counter(t)

# Pattern B: Manual cancel-out (shows understanding)
count = {}
for c in s: count[c] = count.get(c, 0) + 1
for c in t:
    count[c] = count.get(c, 0) - 1
    if count[c] < 0: return False
return True
```